# Transport optimal et Sinkhorn en domaine log, from scratch

**Phase 5 · Géométrie · NB39** · suite directe de **NB36** (fondations géométriques), **NB37** (manifold learning) et **NB38** (optimisation riemannienne).

---

Jusqu’ici, la phase géométrie a surtout demandé où vivent les objets du machine learning. NB36 a posé le vocabulaire des espaces, des métriques et des géodésiques. NB37 a montré comment reconstruire une géométrie inconnue depuis un nuage de points. NB38 a expliqué comment optimiser un paramètre quand il doit rester sur un territoire connu, comme une sphère ou le cône des matrices SPD.

Ici, on change d’objet. On ne compare plus seulement deux points, deux paramètres ou deux matrices. On compare deux **distributions entières** : deux histogrammes, deux nuages pondérés, deux images vues comme une masse répartie sur une grille.

La question du chapitre est donc :

$$
\text{comment comparer deux distributions quand la position de la masse compte ?}
$$

Une distance classique comme $\lVert a-b\rVert_2$, une cross-entropy ou une divergence KL compare surtout les coordonnées au même indice. C’est exactement ce qu’il faut quand les indices sont des événements discrets fixes, par exemple des classes dans une classification, ou quand on régularise un posterior de VAE contre un prior connu. Dans ces cas-là, KL répond à la bonne question :

$$
\text{est-ce que q met de la probabilité là où p en met ?}
$$

Mais cette question devient trop locale quand les indices vivent eux-mêmes dans un espace géométrique. Deux pixels voisins sont proches. Deux bins voisins d’un histogramme sont proches. Deux embeddings voisins dans un latent représentent souvent des états similaires. Si une masse se décale d’un pixel, elle n’a pas disparu pour réapparaître ailleurs sans lien avec son ancienne position : elle a simplement bougé un peu.

Le transport optimal pose alors une autre question :

$$
\text{combien coûte la réorganisation de la masse de a vers b} ?
$$

L’image mentale du notebook sera celle d’un tas de terre à remodeler. La distribution source $a$ dit où se trouve la terre au départ. La distribution cible $b$ dit quelle forme on veut obtenir. Le coût $C_{ij}$ dit combien coûte le déplacement d’une unité de masse depuis le point source $x_i$ vers le point cible $y_j$. Le plan de transport $P_{ij}$ dit combien de masse on envoie réellement de $x_i$ vers $y_j$.

Le cœur mathématique sera donc :

$$
\boxed{
\min_{P\ge 0}\langle P,C\rangle
\quad
\text{sous}
\quad
P\mathbf 1 = a,\qquad P^\top\mathbf 1=b.
}
$$

Les contraintes disent que chaque source envoie toute sa masse et que chaque cible reçoit exactement la masse demandée. La valeur $\langle P,C\rangle$ mesure le coût total du déplacement. Quand $C_{ij}=d(x_i,y_j)^p$, ce coût est relié à la distance de Wasserstein $W_p$. Dans ce notebook, on utilisera souvent un coût quadratique,

$$
C_{ij}=\lVert x_i-y_j\rVert^2,
$$

donc les quantités calculées sont plutôt des coûts de type $W_2^2$ régularisé, pas toujours $W_2$ avec racine.


## Problème du chapitre

Comparer deux distributions comme de simples vecteurs oublie la géométrie du support.

Si deux histogrammes ont le même pic décalé d’un seul bin, une comparaison point à point peut les juger très différents. Pourtant, il suffit de déplacer un peu de masse sur une courte distance. À l’inverse, deux masses de même forme peuvent être éloignées si elles vivent loin l’une de l’autre dans l’espace.

Le transport optimal ajoute précisément cette information manquante :

$$
\text{distribution}
+
\text{géométrie du support}
\longrightarrow
\text{coût de déplacement}.
$$

Il ne remplace donc pas KL, la cross-entropy ou la distance euclidienne partout. Il sert quand la comparaison doit tenir compte de la position de la masse.

| Situation | Distance naturelle |
|---|---|
| Classes discrètes sans géométrie entre labels | cross-entropy, KL |
| Posterior VAE contre prior gaussien | KL analytique |
| Histogrammes dont les bins sont ordonnés | transport optimal utile |
| Images vues comme masse sur une grille | transport optimal utile |
| Deux distributions de latents ou d’embeddings | transport optimal utile si l’on compare des nuages, pas des paires déjà alignées |
| JEPA avec prédiction cible appariée | L2 ou cosine suffisent souvent |
| JEPA avec ensembles de latents non parfaitement appariés | Sinkhorn peut servir à aligner deux distributions de représentations |


## Ce qu’on va construire

| Section | Rôle dans l’histoire |
|---|---|
| 1. Motivation | voir pourquoi comparer bin par bin peut oublier la géométrie |
| 2. Monge et Kantorovich | passer d’une fonction de transport à un plan $P$ qui peut scinder la masse |
| 3. Régularisation entropique | rendre le problème plus lisse et calculable avec Sinkhorn |
| 4. Sinkhorn en domaine log | stabiliser l’algorithme quand $\varepsilon$ est petit ou quand les coûts sont grands |
| 5. Plan de transport | visualiser qui envoie combien de masse vers qui |
| 6. Translation 1D | vérifier que le coût croît quand deux distributions s’éloignent |
| 7. MNIST | transporter la masse d’un chiffre moyen vers un autre sur une vraie grille de pixels |
| 8. Lien génératif | relier cette route primale au WGAN, qui utilise plutôt une route duale |
| 9. Limites et extensions | coût quadratique, biais entropique, Sinkhorn divergence, barycentres |
| 10. Sources | replacer l’algorithme dans la littérature |


## Ce qu’on doit retenir

À la fin du notebook, on veut pouvoir expliquer et coder :

- pourquoi une distribution n’est pas seulement une liste de valeurs, mais une masse posée sur un support ;
- pourquoi KL et cross-entropy restent excellentes quand on compare des probabilités sur les mêmes événements ;
- pourquoi elles peuvent devenir peu informatives quand deux masses sont décalées dans un espace géométrique ;
- comment construire une matrice de coût $C$ à partir des positions du support ;
- ce que représente un plan de transport $P_{ij}$ ;
- pourquoi les contraintes $P\mathbf 1=a$ et $P^\top\mathbf 1=b$ définissent un plan valide ;
- comment la régularisation entropique transforme le programme linéaire en problème résolu par Sinkhorn ;
- pourquoi la version naïve avec $K=\exp(-C/\varepsilon)$ casse numériquement à petit $\varepsilon$ ;
- comment le domaine log et `logsumexp` gardent le calcul stable ;
- pourquoi le plan $P$ est souvent aussi intéressant que le scalaire final.


## Pont avec le génératif

Le NB30 a déjà rencontré Wasserstein avec le WGAN-GP. Ce lien est important, mais il faut le cadrer proprement. Le WGAN vise surtout $W_1$ par la formulation duale de Kantorovich-Rubinstein : il apprend un critic 1-Lipschitz et ne construit jamais le plan de transport.

Dans ce notebook, on prend l’autre route : la route primale. On construit explicitement le plan $P$, sur des distributions discrètes de taille raisonnable. C’est moins adapté aux très grandes images ou aux distributions continues de haute dimension, mais beaucoup plus interprétable : on voit où part la masse.


## Sources principales

- [Cuturi (2013), *Sinkhorn Distances: Lightspeed Computation of Optimal Transport*](https://arxiv.org/abs/1306.0895), pour la régularisation entropique et l’usage moderne de Sinkhorn en transport optimal.
- [Peyré & Cuturi (2019), *Computational Optimal Transport*](https://arxiv.org/abs/1803.00567), pour la référence complète sur Monge, Kantorovich, Wasserstein, Sinkhorn et les extensions.
- [Arjovsky, Chintala & Bottou (2017), *Wasserstein GAN*](https://arxiv.org/abs/1701.07875), pour la route duale utilisée en génératif.
- [Sinkhorn & Knopp (1967), *Concerning nonnegative matrices and doubly stochastic matrices*](https://msp.org/pjm/1967/21-2/pjm-v21-n2-p14-p.pdf), pour l’algorithme de mise à l’échelle alternée.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.special import logsumexp

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import torch
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor

rng = np.random.default_rng(0)
DATA_DIR = Path("../../data")

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
})

px.defaults.template = "plotly_white"

## 1. Motivation : quand comparer case par case ne suffit plus

Avant de parler d’algorithme, il faut sentir le problème.

Supposons que deux distributions vivent sur une ligne. La première met toute sa masse au point $2$ :

$$
a = [0,0,1,0,0,0,0],
$$

et la seconde met toute sa masse juste à côté, au point $3$ :

$$
b_{\text{proche}} = [0,0,0,1,0,0,0].
$$

Une troisième distribution met la même masse beaucoup plus loin :

$$
b_{\text{loin}} = [0,0,0,0,0,0,1].
$$

Visuellement, $b_{\text{proche}}$ est presque la même distribution que $a$, simplement décalée d’un cran. $b_{\text{loin}}$, elle, demande un déplacement beaucoup plus long.

Pourtant, une comparaison point à point voit surtout ceci :

$$
a_i \neq b_i.
$$

Elle regarde chaque case séparément. La masse a disparu d’une case et réapparu dans une autre. Que cette autre case soit voisine ou très loin, la comparaison vectorielle ne le sait pas vraiment.

C’est la limite d’une distance comme

$$
\lVert a-b\rVert_2.
$$

Elle est utile quand les coordonnées sont des features ordinaires ou des bins sans géométrie particulière. Mais si les indices sont des positions dans un espace, pixels d’une image, bins ordonnés d’un histogramme, points d’un nuage, embeddings dans un latent, alors les coordonnées ne sont plus interchangeables. Deux pixels voisins ne devraient pas être traités comme deux pixels aux coins opposés de l’image.

Les divergences probabilistes comme KL ou Jensen-Shannon posent une autre question :

$$
\mathrm{KL}(a\|b)
=
\sum_i a_i \log\frac{a_i}{b_i}.
$$

Elles demandent :

$$
\text{là où } a \text{ met de la masse, est-ce que } b \text{ en met aussi ?}
$$

C’est exactement la bonne question dans beaucoup de notebooks précédents. En classification, la cross-entropy vérifie si la probabilité est sur la bonne classe. Dans un VAE, la KL régularise un posterior latent contre un prior. Là, on compare des probabilités sur les mêmes événements ou deux lois analytiques dans le même système de coordonnées.

Mais pour deux masses posées sur un support géométrique, cette question est parfois trop stricte. Si une masse se décale légèrement, KL peut la voir comme une disparition puis une réapparition, au lieu de voir un petit déplacement. Dans le cas extrême de deux masses de Dirac en deux points différents, les supports ne se recouvrent pas, et $\mathrm{KL}(a\|b)$ peut devenir infinie, que les deux points soient voisins ou très éloignés.

Le transport optimal change donc la question.

Au lieu de demander :

$$
\text{les deux distributions mettent-elles leur masse exactement aux mêmes endroits ?}
$$

on demande :

$$
\text{combien coûte le déplacement de la masse de } a \text{ pour obtenir } b ?
$$

Cette fois, l’espace compte. Déplacer une unité de masse vers un point voisin coûte peu. La déplacer à l’autre bout du support coûte plus cher. La distribution n’est plus seulement une liste de nombres : c’est une masse posée sur un territoire.

Le transport optimal ajoute donc une information que les comparaisons case par case ignorent :

$$
\boxed{
\text{valeurs de la distribution}
+
\text{géométrie du support}
\longrightarrow
\text{coût de réorganisation de la masse}.
}
$$

Dans ce notebook, cette idée va devenir concrète. On va construire une matrice de coût $C$, puis un plan de transport $P$. Le coût dira combien coûte un déplacement élémentaire. Le plan dira combien de masse part de chaque point source vers chaque point cible.


## 2. Monge et Kantorovich : du déplacement unique au plan de masse

On commence avec deux distributions discrètes :

$$
a = (a_1,\ldots,a_n),
\qquad
b = (b_1,\ldots,b_m).
$$

Chaque $a_i$ est une masse placée sur un point source $x_i$. Chaque $b_j$ est une masse demandée sur un point cible $y_j$. On suppose que les deux distributions ont la même masse totale :

$$
a_i \ge 0,\qquad b_j \ge 0,
\qquad
\sum_i a_i = \sum_j b_j = 1.
$$

La géométrie du support entre ensuite par une matrice de coût :

$$
C_{ij} = c(x_i,y_j).
$$

Par exemple, si les points vivent dans un espace euclidien, on peut prendre :

$$
C_{ij} = \lVert x_i-y_j\rVert
$$

ou, comme on le fera souvent dans ce notebook,

$$
C_{ij} = \lVert x_i-y_j\rVert^2.
$$

Le coût $C_{ij}$ ne dit pas encore combien de masse on déplace. Il dit seulement le prix d’un déplacement élémentaire : envoyer une unité de masse de $x_i$ vers $y_j$.


### 2.1 Le problème de Monge

Historiquement, Monge formule le problème comme une fonction de transport. Chaque point source $x_i$ choisit une destination unique :

$$
T(x_i) = y_j.
$$

On cherche alors la fonction $T$ qui minimise le coût total :

$$
\min_T \sum_i a_i\, C(x_i, T(x_i))
\qquad
\text{sous la contrainte}
\qquad
T_\# a = b.
$$

La notation $T_\# a = b$ signifie que si l’on pousse toute la masse de $a$ à travers $T$, on obtient exactement la distribution cible $b$.

Cette formulation est très naturelle. Elle ressemble à une instruction physique : "chaque tas de terre part vers une destination". Mais elle est trop rigide.

Si un point source contient beaucoup de masse et que plusieurs points cibles doivent en recevoir une partie, une fonction $T$ ne peut pas le faire. Une fonction envoie un point vers un seul point. Elle ne sait pas scinder la masse.

C’est le blocage principal : Monge cherche un transport déterministe, alors que le problème naturel demande parfois de partager une même source entre plusieurs destinations.


### 2.2 La relaxation de Kantorovich

Kantorovich remplace la fonction $T$ par un plan de transport :

$$
P_{ij} \ge 0.
$$

Cette fois, $P_{ij}$ désigne la quantité de masse envoyée de la source $x_i$ vers la cible $y_j$.

Le plan peut scinder la masse. Une ligne de $P$ peut envoyer une source vers plusieurs cibles. Une colonne peut recevoir de plusieurs sources. C’est exactement ce qu’il faut pour manipuler des distributions discrètes.

Un plan est valide s’il respecte deux contraintes.

D’abord, chaque source doit envoyer toute sa masse :

$$
P\mathbf 1_m = a.
$$

Autrement dit, la somme de la ligne $i$ vaut $a_i$ :

$$
\sum_j P_{ij}=a_i.
$$

Ensuite, chaque cible doit recevoir la bonne masse :

$$
P^\top\mathbf 1_n = b,
$$

c’est-à-dire :

$$
\sum_i P_{ij}=b_j.
$$

L’ensemble des plans valides est souvent noté :

$$
U(a,b)
=
\{P\in\mathbb R_+^{n\times m}
:
P\mathbf 1_m=a,\,
P^\top\mathbf 1_n=b
\}.
$$
> Ici, $\mathbf 1_m$ désigne un vecteur de $m$ uns : multiplier $P$ par $\mathbf 1_m$ somme chaque ligne de $P$, tandis que multiplier $P^\top$ par $\mathbf 1_n$ somme chaque colonne de $P$.

Une fois ce vocabulaire posé, le problème de transport optimal discret devient :

$$
\boxed{
\min_{P\in U(a,b)}
\langle P,C\rangle
=
\min_{P\in U(a,b)}
\sum_{i,j}P_{ij}C_{ij}.
}
$$

Cette formule dit simplement : parmi tous les plans qui transforment bien $a$ en $b$, choisir celui dont le coût total est minimal.

Le produit $P_{ij}C_{ij}$ a une lecture directe :

$$
\text{masse envoyée de } i \text{ vers } j
\times
\text{prix unitaire du trajet } i\to j.
$$

En sommant sur tous les trajets, on obtient le coût total du chantier.

Quand le coût est

$$
C_{ij}=d(x_i,y_j)^p,
$$

la valeur optimale correspond à $W_p^p(a,b)$. La distance de Wasserstein elle-même est donc :

$$
W_p(a,b)
=
\left(
\min_{P\in U(a,b)}
\sum_{i,j}P_{ij}d(x_i,y_j)^p
\right)^{1/p}.
$$

Dans la suite, quand on utilisera un coût quadratique $C_{ij}=\lVert x_i-y_j\rVert^2$, le scalaire $\langle P,C\rangle$ sera donc un coût de type $W_2^2$. C’est important : on ne prendra pas toujours la racine carrée, et la régularisation entropique introduite ensuite modifiera encore légèrement la valeur.
On choisit ici le coût quadratique euclidien parce que nos premiers supports sont des espaces euclidiens simples : une ligne, un plan, puis une grille de pixels MNIST. Ce choix n’est pas une hypothèse générale du transport optimal. Toute la géométrie du problème entre par la matrice $C$ : sur une variété, on remplacerait naturellement $\lVert x_i-y_j\rVert^2$ par une distance géodésique $d_{\mathcal M}(x_i,y_j)^2$, une distance de graphe, ou une distance adaptée au latent étudié.

```mermaid
flowchart LR
    A["Source a<br/>masses a_i sur les points x_i"]
    B["Cible b<br/>masses b_j sur les points y_j"]
    C["Coût C_ij<br/>prix pour déplacer une unité de x_i vers y_j"]
    P["Plan P_ij<br/>quantité envoyée de x_i vers y_j"]
    V["Contraintes<br/>P 1 = a<br/>P^T 1 = b"]
    S["Coût total<br/>somme P_ij C_ij"]

    A --> P
    B --> P
    C --> P
    P --> V
    P --> S

    classDef default fill:none,stroke:#64748b,stroke-width:1.5px;
    classDef focus fill:none,stroke:#16a34a,stroke-width:2px;
    class P,S focus;
```

**Lecture :** $a$, $b$ et $C$ sont les données du problème. Le plan $P$ est l’inconnue. Les contraintes de marginales garantissent que $P$ transporte bien toute la masse source vers la masse cible. L’objectif $\langle P,C\rangle$ choisit, parmi tous les plans valides, celui qui dépense le moins.

Cette formulation est propre et convexe. Elle a cependant un coût pratique : résoudre exactement le programme linéaire devient vite lourd quand le nombre de points augmente, et la solution exacte peut changer brutalement quand $a$, $b$ ou $C$ bougent. Pour un notebook from scratch et pour des usages différentiables en machine learning, on veut une version plus lisse et plus rapide à calculer.

C’est le rôle de la régularisation entropique et de Sinkhorn.


In [ ]:
def transport_cost_matrix(X, Y):
    """Matrice de coût quadratique C_ij = ||x_i - y_j||^2.

    X contient les n positions source x_i.
    Y contient les m positions cible y_j.
    Le résultat C a shape (n, m), avec une ligne par source et une colonne par cible.
    """
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)

    # Identité utile :
    # ||x-y||^2 = ||x||^2 + ||y||^2 - 2 <x,y>
    # Elle évite une double boucle Python sur les paires (i, j).
    sq_X = np.sum(X ** 2, axis=1)[:, None]
    sq_Y = np.sum(Y ** 2, axis=1)[None, :]
    C = sq_X + sq_Y - 2.0 * X @ Y.T

    # De très petits négatifs peuvent apparaître par arrondi flottant.
    return np.maximum(C, 0.0)


# Mini-test : deux points identiques coûtent 0 ; le triangle 3-4-5 coûte 5^2 = 25.
X_test = np.array([[0.0, 0.0], [3.0, 4.0]])
Y_test = np.array([[0.0, 0.0], [0.0, 0.0]])

C_test = transport_cost_matrix(X_test, Y_test)

assert C_test.shape == (2, 2)
assert np.allclose(C_test[0], [0.0, 0.0])
assert np.allclose(C_test[1], [25.0, 25.0])

print("transport_cost_matrix OK")
print(C_test)

Avant de chercher un plan de transport, il faut fixer la géométrie du support. Ici on choisit le coût quadratique

$$
C_{ij}=\lVert x_i-y_j\rVert^2.
$$

Cela signifie que le scalaire $\langle P,C\rangle$ se lit comme un coût de type $W_2^2$ : déplacer une masse deux fois plus loin coûte quatre fois plus cher.

## 3. Régularisation entropique : rendre le transport calculable

À ce stade, on sait formuler le bon problème :

$$
\min_{P\in U(a,b)} \langle P,C\rangle.
$$

Le plan $P$ doit respecter les marginales, et parmi tous les plans valides on veut celui qui coûte le moins cher. Mathématiquement, c’est propre. Mais calculatoirement, c’est lourd : on résout un programme linéaire avec $n\times m$ variables, une contrainte par source, une contrainte par cible, et une contrainte de positivité sur chaque coefficient.

Il y a aussi un problème plus subtil. Le plan optimal exact est souvent très concentré. Beaucoup de coefficients valent zéro, et une petite variation de $a$, $b$ ou $C$ peut changer brutalement quels trajets sont actifs. Pour une analyse géométrique, ce n’est pas forcément grave. Pour une loss différentiable dans un modèle de machine learning, c’est moins confortable.

Cuturi (2013) propose alors une idée simple : au lieu de chercher directement le plan le moins cher, on cherche un plan presque aussi bon, mais un peu plus diffus.

On ajoute une régularisation entropique :

$$
\min_{P\in U(a,b)}
\langle P,C\rangle
+
\varepsilon \sum_{i,j} P_{ij}(\log P_{ij}-1).
$$

Dans le code, on appellera ce paramètre `reg`. Dans les équations, on peut le lire comme $\varepsilon$.

Le premier terme veut un plan peu coûteux :

$$
\langle P,C\rangle
=
\sum_{i,j}P_{ij}C_{ij}.
$$

Le second terme évite les plans trop secs, trop concentrés sur quelques trajets. Il pousse le plan à garder un peu d’entropie, donc à répartir la masse entre plusieurs destinations raisonnables plutôt que choisir immédiatement un plan très dur.

L’image mentale est utile. Sans entropie, le transport optimal cherche le plan le plus économique, quitte à faire des décisions très tranchées. Avec entropie, on accepte un plan un peu plus flou, comme si le transport gardait une petite température :

| Valeur de $\varepsilon$ | Effet sur le plan |
|---|---|
| $\varepsilon$ grand | plan diffus, beaucoup de trajets reçoivent un peu de masse |
| $\varepsilon$ petit | plan plus net, plus proche du transport optimal exact |
| $\varepsilon \to 0$ | on revient vers le problème non régularisé, mais le calcul devient plus difficile |

Il faut donc voir $\varepsilon$ comme un compromis. Trop grand, le plan est stable mais trop flou. Trop petit, le plan ressemble mieux au vrai transport optimal, mais le calcul devient instable.

La magie de cette régularisation est qu’elle change la forme de la solution. Le plan optimal régularisé peut s’écrire comme une matrice de préférences géométriques, corrigée par deux facteurs d’échelle :

$$
P_{ij}
=
u_i K_{ij} v_j,
\qquad
K_{ij}=\exp\left(-\frac{C_{ij}}{\varepsilon}\right).
$$

Le noyau $K$ dépend seulement du coût. Si $C_{ij}$ est petit, alors $K_{ij}$ est grand : déplacer de $i$ vers $j$ est plausible. Si $C_{ij}$ est grand, alors $K_{ij}$ est proche de zéro : ce trajet est cher, donc peu utilisé.

Mais $K$ seul ne respecte pas les masses $a$ et $b$. Il dit seulement quels trajets sont naturellement bon marché. Les vecteurs $u$ et $v$ servent à corriger les lignes et les colonnes pour forcer les bonnes marginales :

$$
P\mathbf 1 = a,
\qquad
P^\top\mathbf 1=b.
$$

On peut lire Sinkhorn comme une alternance très concrète :

1. corriger les lignes pour que chaque source envoie la bonne masse ;
2. corriger les colonnes pour que chaque cible reçoive la bonne masse ;
3. recommencer jusqu’à ce que les deux contraintes soient satisfaites.

Avec la forme $P=\operatorname{diag}(u)K\operatorname{diag}(v)$, les mises à jour deviennent :

$$
u \leftarrow \frac{a}{Kv},
\qquad
v \leftarrow \frac{b}{K^\top u}.
$$

Les divisions sont élément par élément. Chaque itération coûte essentiellement un produit matrice-vecteur avec $K$, donc $O(nm)$ par itération. C’est beaucoup plus simple à coder qu’un solveur de programme linéaire général.

Dans notre implémentation log-domain, on utilisera une forme équivalente avec une mesure de référence $a_i b_j$ :

$$
P_{ij}
=
\exp\left(\frac{f_i+g_j-C_{ij}}{\varepsilon}\right)a_i b_j.
$$

Les termes $a_i b_j$ ne changent pas l’idée du problème : sous les contraintes de marginales, ils correspondent à une écriture équivalente de la régularisation. Ils rendent surtout les mises à jour log-domain propres et cohérentes avec les masses nulles.

Le point à retenir est celui-ci :

$$
\boxed{
\text{l'entropie transforme un programme linéaire difficile en un problème de mise à l'échelle de lignes et de colonnes.}
}
$$

Le prix à payer est le biais entropique. Le coût retourné par Sinkhorn n’est pas exactement la distance de Wasserstein non régularisée. C’est le coût d’un plan régularisé, plus lisse, plus stable, et beaucoup plus facile à calculer.


## 4. Sinkhorn en domaine log : le même calcul, mais stable


### 4.1 Pourquoi la version naïve casse

La version naïve de Sinkhorn commence par construire

$$
K_{ij}
=
\exp\left(-\frac{C_{ij}}{\varepsilon}\right).
$$

Cette formule a l’air innocente. Elle dit simplement : plus un trajet coûte cher, moins on veut l’utiliser.

Mais numériquement, elle est dangereuse. Si $\varepsilon$ est petit, le quotient

$$
\frac{C_{ij}}{\varepsilon}
$$

peut devenir très grand. L’exposant

$$
-\frac{C_{ij}}{\varepsilon}
$$

devient alors très négatif. En float64, `np.exp(x)` sous-flotte vers zéro autour de $x\approx -745$.

Donc si

$$
\frac{C_{ij}}{\varepsilon} \gtrsim 745,
$$

l’entrée $K_{ij}$ devient exactement `0.0`.

Ce n’est pas seulement une perte de précision. Sinkhorn doit ensuite faire :

$$
u \leftarrow \frac{a}{Kv}.
$$

Si une ligne de $K$ est devenue presque toute nulle, le dénominateur $(Kv)_i$ peut valoir zéro. On divise alors par zéro, puis les `inf` et les `NaN` contaminent le plan.

C’est précisément le régime qu’on veut pourtant atteindre : petit $\varepsilon$, plan plus net, transport plus proche du problème non régularisé. Le calcul naïf échoue au moment où il devient intéressant.


### 4.2 L’idée du domaine log

Le remède est de ne pas manipuler directement les facteurs d’échelle $u$ et $v$.

On suit plutôt leurs logarithmes, mis à l’échelle par $\varepsilon$ :

$$
f = \varepsilon \log u,
\qquad
g = \varepsilon \log v.
$$

Les vecteurs $f$ et $g$ sont appelés des potentiels duaux. Pour l’intuition, on peut les voir comme des corrections de lignes et de colonnes : $f_i$ ajuste la source $i$, et $g_j$ ajuste la cible $j$, pour que le plan respecte les marginales.

Avec ces potentiels, le plan s’écrit :

$$
P_{ij}
=
\exp\left(
\frac{f_i+g_j-C_{ij}}{\varepsilon}
\right)
a_i b_j.
$$

La quantité dans l’exponentielle garde la même logique qu’avant :

$$
f_i + g_j - C_{ij}.
$$

Un trajet cher, grand $C_{ij}$, est pénalisé. Les potentiels $f_i$ et $g_j$ rééquilibrent les lignes et les colonnes pour satisfaire les masses imposées.

Le problème est qu’il faut encore sommer des exponentielles. Par exemple, pour forcer la ligne $i$ à sommer vers $a_i$, on doit calculer une quantité du type :

$$
\sum_j
\exp\left(
\frac{g_j-C_{ij}}{\varepsilon}
\right)b_j.
$$

Au lieu de calculer cette somme directement, on calcule son logarithme avec `logsumexp`.

La fonction `logsumexp` utilise l’identité :

$$
\log\sum_j \exp(z_j)
=
M + \log\sum_j \exp(z_j-M),
\qquad
M=\max_j z_j.
$$

En soustrayant le maximum avant l’exponentielle, on évite de calculer des exponentielles trop grandes ou trop petites. Le résultat mathématique est le même, mais le calcul reste fini.


### 4.3 Les mises à jour log-domain

Les mises à jour deviennent :

$$
\boxed{
\begin{aligned}
&\textbf{initialiser } f \gets 0,\quad g \gets 0 \\
&\textbf{répéter} \\
&\quad f_i \gets
-\varepsilon
\log\sum_j
\exp\left(
\frac{g_j-C_{ij}}{\varepsilon}
+
\log b_j
\right)
\\
&\quad g_j \gets
-\varepsilon
\log\sum_i
\exp\left(
\frac{f_i-C_{ij}}{\varepsilon}
+
\log a_i
\right)
\\
&\textbf{jusqu'à ce que }
\lVert P\mathbf 1-a\rVert_1 < \text{tol}
\\
&\quad
P_{ij}
=
\exp\left(
\frac{f_i+g_j-C_{ij}}{\varepsilon}
\right)
a_i b_j.
\end{aligned}
}
$$

Ces formules peuvent paraître plus lourdes que les mises à jour naïves

$$
u \leftarrow \frac{a}{Kv},
\qquad
v \leftarrow \frac{b}{K^\top u}.
$$

Mais elles font le même travail : alterner entre correction des lignes et correction des colonnes. La seule différence est numérique. La version naïve construit $K=\exp(-C/\varepsilon)$ directement. La version log-domain garde les calculs dans les logarithmes aussi longtemps que possible.

C’est le même Sinkhorn, mais écrit dans une coordonnée où les nombres ne s’effondrent pas.


### 4.4 Ce qu’on va vérifier

La suite du notebook va donc tester deux choses.

D’abord, dans un régime confortable, la version naïve et la version log-domain doivent donner le même plan. C’est important : le domaine log n’est pas un autre algorithme, c’est une autre écriture du même calcul.

Ensuite, quand $\varepsilon$ devient petit, la version naïve doit casser, tandis que la version log-domain doit rester finie et respecter les marginales :

$$
P\mathbf 1 \approx a,
\qquad
P^\top\mathbf 1\approx b.
$$

C’est le vrai contrat d’un plan de transport calculé par Sinkhorn : pas seulement produire un nombre, mais produire une matrice $P$ qui transporte bien toute la masse source vers toute la masse cible.


In [ ]:
def sinkhorn_naive(a, b, C, reg, n_iters=200):
    """Sinkhorn-Knopp naïf : construit directement K = exp(-C/reg).

    Cette version sert de contre-exemple pédagogique. Elle fonctionne quand
    reg est assez grand, mais devient instable dès que C/reg est trop grand.
    """
    K = np.exp(-C / reg)

    u = np.ones_like(a)
    v = np.ones_like(b)

    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        for _ in range(n_iters):
            u = a / (K @ v)
            v = b / (K.T @ u)

    P = u[:, None] * K * v[None, :]
    return P


def sinkhorn_log_domain(a, b, C, reg, n_iters=1000, tol=1e-9):
    """Sinkhorn en domaine log, avec potentiels duaux f et g.

    Résout le problème de transport entropique sans former explicitement
    K = exp(-C/reg). Les lignes de P doivent sommer vers a, et les colonnes
    vers b.
    """
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    C = np.asarray(C, dtype=np.float64)

    if reg <= 0:
        raise ValueError("reg doit être strictement positif.")

    with np.errstate(divide="ignore"):
        log_a = np.log(a)
        log_b = np.log(b)

    f = np.zeros_like(a)
    g = np.zeros_like(b)
    neg_C_over_reg = -C / reg

    for _ in range(n_iters):
        f = -reg * logsumexp(
            neg_C_over_reg + (g / reg + log_b)[None, :],
            axis=1,
        )
        g = -reg * logsumexp(
            neg_C_over_reg + (f / reg + log_a)[:, None],
            axis=0,
        )

        log_P = (
            (f[:, None] + g[None, :]) / reg
            + neg_C_over_reg
            + log_a[:, None]
            + log_b[None, :]
        )
        P = np.exp(log_P)

        row_error = np.sum(np.abs(P.sum(axis=1) - a))
        col_error = np.sum(np.abs(P.sum(axis=0) - b))
        if max(row_error, col_error) < tol:
            break

    return P, f, g

La première fonction est volontairement fragile : elle matérialise $K=\exp(-C/\varepsilon)$, ce qui permet de voir pourquoi la version naïve casse. La seconde fait le même Sinkhorn, mais dans les variables log-domain $f$ et $g$ ; elle évite de former directement un noyau $K$ rempli de zéros numériques.

### 4.3 Mini-test : un plan valide retrouve ses marginales

Avant de parler de coût optimal, vérifions d’abord le contrat minimal d’un plan de transport.

Une matrice $P$ peut avoir la bonne taille et des valeurs positives sans être un plan valide. Pour transporter $a$ vers $b$, elle doit respecter les marginales :

$$
P\mathbf 1 = a,
\qquad
P^\top\mathbf 1 = b.
$$

On teste donc Sinkhorn sur un petit exemple synthétique. Le paramètre `reg = 0.1` est volontairement confortable : dans ce régime, le noyau $K=\exp(-C/\mathrm{reg})$ ne sous-flotte pas encore, donc la version naïve et la version log-domain doivent donner le même résultat.

In [ ]:
n_small, m_small = 6, 5

X_small = rng.normal(size=(n_small, 2))
Y_small = rng.normal(loc=2.0, size=(m_small, 2))

a_small = np.full(n_small, 1.0 / n_small)
b_small = np.full(m_small, 1.0 / m_small)
C_small = transport_cost_matrix(X_small, Y_small)

P_log, f_small, g_small = sinkhorn_log_domain(
    a_small,
    b_small,
    C_small,
    reg=0.1,
)
P_naive = sinkhorn_naive(
    a_small,
    b_small,
    C_small,
    reg=0.1,
)

row_residual = np.abs(P_log.sum(axis=1) - a_small).max()
col_residual = np.abs(P_log.sum(axis=0) - b_small).max()
total_mass = P_log.sum()

assert row_residual < 1e-6, f"marginale ligne incorrecte : {row_residual:.2e}"
assert col_residual < 1e-6, f"marginale colonne incorrecte : {col_residual:.2e}"
assert np.isclose(total_mass, 1.0)
assert np.allclose(P_log, P_naive, atol=1e-6)

print(f"Plan shape              : {P_log.shape}")
print(f"Masse totale de P       : {total_mass:.6f}")
print(f"max|P.sum(axis=1) - a|  : {row_residual:.2e}")
print(f"max|P.sum(axis=0) - b|  : {col_residual:.2e}")
print("Naïf et log-domain concordent à reg = 0.1.")

**Lecture :** ce test ne prouve pas encore que le plan est optimal. Il vérifie le contrat de transport : toute la masse source est envoyée, toute la masse cible est reçue, et la masse totale reste égale à $1$. Dans un régime numérique facile, la version naïve et la version log-domain coïncident, ce qui confirme que le domaine log ne change pas l’algorithme. Il change seulement la façon de le calculer.

### 4.4 Le régime où la version naïve casse

On force maintenant le problème numérique décrit plus haut.

Avec `reg = 0.01`, le plan régularisé devrait devenir plus net, donc plus proche du transport non régularisé. Mais le noyau naïf

$$
K_{ij}=\exp(-C_{ij}/\mathrm{reg})
$$

devient dangereux : si $C_{ij}/\mathrm{reg}$ dépasse environ $745$, `np.exp(-C_ij / reg)` tombe à zéro en float64. Une ligne de $K$ peut alors contenir trop de zéros, et la mise à jour

$$
u \leftarrow \frac{a}{Kv}
$$

rencontre un dénominateur nul.

In [ ]:
reg_tiny = 0.01

P_naive_tiny = sinkhorn_naive(
    a_small,
    b_small,
    C_small,
    reg=reg_tiny,
)
P_log_tiny, f_tiny, g_tiny = sinkhorn_log_domain(
    a_small,
    b_small,
    C_small,
    reg=reg_tiny,
)

K_tiny = np.exp(-C_small / reg_tiny)

n_naive_nans = int(np.isnan(P_naive_tiny).sum())
n_log_nans = int(np.isnan(P_log_tiny).sum())

row_residual_tiny = np.abs(P_log_tiny.sum(axis=1) - a_small).max()
col_residual_tiny = np.abs(P_log_tiny.sum(axis=0) - b_small).max()

print(f"reg                         : {reg_tiny}")
print(f"max(C/reg)                  : {(C_small / reg_tiny).max():.1f}")
print(f"K.min(), K.max()             : {K_tiny.min():.3e}, {K_tiny.max():.3e}")
print(f"entrées nulles dans K        : {np.sum(K_tiny == 0.0)}/{K_tiny.size}")
print(f"NaN dans Sinkhorn naïf       : {n_naive_nans}/{P_naive_tiny.size}")
print(f"NaN dans Sinkhorn log-domain : {n_log_nans}/{P_log_tiny.size}")
print(f"max|P.sum(axis=1) - a|       : {row_residual_tiny:.2e}")
print(f"max|P.sum(axis=0) - b|       : {col_residual_tiny:.2e}")

assert n_naive_nans > 0
assert n_log_nans == 0
assert np.all(np.isfinite(P_log_tiny))
assert row_residual_tiny < 1e-6
assert col_residual_tiny < 1e-6

print("\nDémonstration confirmée : le naïf casse, le log-domain reste valide.")

**Lecture :** la version naïve échoue parce qu’elle matérialise directement $K=\exp(-C/\mathrm{reg})$. À petit `reg`, beaucoup d’entrées de $K$ deviennent exactement nulles en float64. Les divisions de Sinkhorn rencontrent alors des zéros, puis les `NaN` se propagent.

La version log-domain évite ce piège. Elle calcule les mêmes corrections de lignes et de colonnes, mais elle garde les sommes sous forme de `logsumexp` aussi longtemps que possible. Le résultat reste fini et respecte encore les deux marginales :

$$
P\mathbf 1 \approx a,
\qquad
P^\top\mathbf 1 \approx b.
$$

Le message pratique n’est pas que la version naïve est toujours fausse : elle peut marcher quand `reg` est grand et que les coûts sont bien échelonnés. Le message est plus précis : dès qu’on veut un plan net, ou que l’échelle de $C$ est grande, il faut stabiliser Sinkhorn. Le domaine log est la manière la plus directe de le faire ici.

### 4.5 Assembler coût, plan et scalaire de transport

Pour la suite, on regroupe la chaîne utile dans une petite fonction :

$$
(a,b,C)
\longrightarrow
P_{\varepsilon}
\longrightarrow
\langle P_{\varepsilon}, C\rangle.
$$

Le scalaire retourné n’est pas exactement la distance de Wasserstein non régularisée. C’est le coût brut du plan entropique calculé par Sinkhorn :

$$
\sum_{i,j} P_{\varepsilon,ij} C_{ij}.
$$

Si $C_{ij}=\lVert x_i-y_j\rVert^2$, ce coût se lit comme une approximation régularisée de type $W_2^2$. On renvoie aussi le plan $P_\varepsilon$, parce que dans ce notebook le plan est aussi important que le nombre final : il montre où va la masse.

In [ ]:
def sinkhorn_transport_cost(a, b, C, reg=0.1, n_iters=1000, tol=1e-9):
    """Coût brut <P, C> du plan de Sinkhorn régularisé.

    Retourne le scalaire de transport et le plan P. Le scalaire est le coût
    du plan entropique, pas la distance de Wasserstein exacte non régularisée.
    """
    P, _, _ = sinkhorn_log_domain(
        a,
        b,
        C,
        reg=reg,
        n_iters=n_iters,
        tol=tol,
    )
    cost = float(np.sum(P * C))
    return cost, P


# Mini-test : transporter une distribution vers elle-même coûte presque zéro
# quand le support est identique et que le coût diagonal vaut zéro.
C_self = transport_cost_matrix(X_small, X_small)

self_cost_loose, _ = sinkhorn_transport_cost(
    a_small,
    a_small,
    C_self,
    reg=0.05,
)
self_cost_sharp, P_self = sinkhorn_transport_cost(
    a_small,
    a_small,
    C_self,
    reg=0.01,
)

assert self_cost_sharp >= 0.0
assert self_cost_sharp < self_cost_loose
assert np.allclose(P_self.sum(axis=1), a_small, atol=1e-6)
assert np.allclose(P_self.sum(axis=0), a_small, atol=1e-6)

print(f"Coût à soi-même, reg=0.05 : {self_cost_loose:.2e}")
print(f"Coût à soi-même, reg=0.01 : {self_cost_sharp:.2e}")
print("Quand reg diminue, le plan se concentre davantage près de la diagonale.")

Même quand $a=b$, le coût régularisé n’est pas forcément exactement nul : l’entropie diffuse un peu le plan hors de la diagonale. En diminuant `reg`, le plan devient plus net et le coût à soi-même se rapproche de zéro.

In [ ]:
def wasserstein_distance(a, b, C, reg=0.1, n_iters=1000, tol=1e-9):
    """Coût <P, C> du plan Sinkhorn, approximation régularisée de Wasserstein."""
    P, _, _ = sinkhorn_log_domain(a, b, C, reg=reg, n_iters=n_iters, tol=tol)
    return float(np.sum(P * C)), P

## 5. Visualiser le plan : le scalaire ne raconte pas tout

Le coût de transport est utile, mais il écrase beaucoup d’information dans un seul nombre.

Le plan $P$ est plus riche. Il dit, pour chaque paire source-cible :

$$
P_{ij}
=
\text{masse envoyée de } x_i \text{ vers } y_j.
$$

On construit un exemple volontairement lisible : une source en forme d’anneau et une cible en forme de blob décalé vers la droite. Le plan ne va pas simplement relier chaque source à son point cible le plus proche. Il doit minimiser un coût total tout en respectant les masses : chaque source envoie exactement ce qu’elle possède, chaque cible reçoit exactement ce qu’elle demande. La figure sert justement à voir ce compromis global.

La heatmap de $P$ montrera la matrice complète. Les segments dans le plan montreront seulement les trajets les plus importants, pour garder une figure lisible.

In [ ]:
n_pts, m_pts = 24, 20

theta = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
X_ring = np.stack(
    [2.0 * np.cos(theta), 2.0 * np.sin(theta)],
    axis=1,
)
X_ring += rng.normal(scale=0.08, size=(n_pts, 2))

Y_blob = rng.normal(
    loc=[4.5, 0.0],
    scale=0.6,
    size=(m_pts, 2),
)

a_ring = np.full(n_pts, 1.0 / n_pts)
b_blob = np.full(m_pts, 1.0 / m_pts)

C_ring_blob = transport_cost_matrix(X_ring, Y_blob)

cost_ring_blob, P_ring_blob = sinkhorn_transport_cost(
    a_ring,
    b_blob,
    C_ring_blob,
    reg=0.05,
)

row_error = np.abs(P_ring_blob.sum(axis=1) - a_ring).max()
col_error = np.abs(P_ring_blob.sum(axis=0) - b_blob).max()

print(f"Coût Sinkhorn(anneau, blob) : {cost_ring_blob:.3f}")
print(f"Shape du plan P             : {P_ring_blob.shape}")
print(f"max|P.sum(axis=1) - a|      : {row_error:.2e}")
print(f"max|P.sum(axis=0) - b|      : {col_error:.2e}")

In [ ]:
threshold = 0.06 * P_ring_blob.max()

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Trajets principaux du plan P",
        "Heatmap du plan P_ij",
    ),
    specs=[[{"type": "xy"}, {"type": "heatmap"}]],
    horizontal_spacing=0.12,
)

fig.add_trace(
    go.Scatter(
        x=X_ring[:, 0],
        y=X_ring[:, 1],
        mode="markers",
        name="source a (anneau)",
        marker=dict(color="#2563eb", size=9, line=dict(color="white", width=1)),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=Y_blob[:, 0],
        y=Y_blob[:, 1],
        mode="markers",
        name="cible b (blob)",
        marker=dict(color="#dc2626", size=9, line=dict(color="white", width=1)),
    ),
    row=1,
    col=1,
)

for i in range(n_pts):
    for j in range(m_pts):
        mass = P_ring_blob[i, j]
        if mass >= threshold:
            opacity = 0.15 + 0.75 * mass / P_ring_blob.max()
            fig.add_trace(
                go.Scatter(
                    x=[X_ring[i, 0], Y_blob[j, 0]],
                    y=[X_ring[i, 1], Y_blob[j, 1]],
                    mode="lines",
                    line=dict(color=f"rgba(80,80,80,{opacity:.3f})", width=1.5),
                    showlegend=False,
                    hoverinfo="skip",
                ),
                row=1,
                col=1,
            )

fig.add_trace(
    go.Heatmap(
        z=P_ring_blob,
        colorscale="Viridis",
        colorbar=dict(title="masse"),
        hovertemplate="source i=%{y}<br>cible j=%{x}<br>P_ij=%{z:.4f}<extra></extra>",
    ),
    row=1,
    col=2,
)

fig.update_xaxes(title_text="coordonnée x", row=1, col=1)
fig.update_yaxes(title_text="coordonnée y", scaleanchor="x", scaleratio=1, row=1, col=1)

fig.update_xaxes(title_text="indice cible j", row=1, col=2)
fig.update_yaxes(title_text="indice source i", autorange="reversed", row=1, col=2)

fig.update_layout(
    width=1000,
    height=460,
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="left", x=0.0),
    margin=dict(l=30, r=30, t=80, b=40),
)

fig.show()

**Lecture :** le plan $P$ n’est pas une collection de plus proches voisins. C’est une solution globale sous contraintes de masse.

Dans le panneau de gauche, chaque segment représente un trajet source-cible dont la masse transportée est assez grande pour être visible. Les trajets courts sont favorisés parce qu’ils coûtent peu, mais ils ne suffisent pas à eux seuls : chaque source doit envoyer exactement sa masse $a_i$, et chaque cible doit recevoir exactement sa masse $b_j$. Le plan équilibre donc deux forces : éviter les longs déplacements, tout en remplissant correctement toute la distribution cible.

La heatmap montre cette négociation sous forme matricielle. Une ligne correspond à une source $i$, une colonne à une cible $j$, et la couleur indique la masse $P_{ij}$. Comme on utilise une régularisation entropique, une source peut répartir sa masse sur plusieurs cibles plausibles au lieu de choisir un unique trajet dur.

Le nombre `cost_ring_blob` résume le coût total, mais il ne dit pas comment ce compromis est construit. Le plan, lui, montre la réorganisation de la masse.

## 6. Test de translation : le coût suit le déplacement de la masse

On veut maintenant vérifier une propriété simple : quand deux distributions ont la même forme mais qu’on les éloigne dans l’espace, le coût de transport doit augmenter avec la distance parcourue.

On prend deux gaussiennes 1D de même variance. La première reste centrée en $0$ :

$$
a \approx \mathcal N(0,\sigma^2),
$$

et la seconde est translatée de $\delta$ :

$$
b_\delta \approx \mathcal N(\delta,\sigma^2).
$$

Les deux distributions ont la même forme. La seule différence est leur position. Le transport optimal doit donc mesurer le travail nécessaire pour déplacer toute la masse de $\delta$ unités.

Pour deux gaussiennes continues de même variance, avec coût quadratique, on connaît la valeur exacte :

$$
W_2^2
\left(
\mathcal N(\mu_1,\sigma^2),
\mathcal N(\mu_2,\sigma^2)
\right)
=
(\mu_1-\mu_2)^2.
$$

Ici, on travaille sur une grille discrète et avec un plan Sinkhorn régularisé. On ne s’attend donc pas à retomber exactement sur $\delta^2$, mais on veut retrouver la bonne tendance : plus $\delta$ grandit, plus le coût augmente.

Cette expérience ne dit pas que KL serait mauvaise pour deux gaussiennes. Au contraire, dans ce cas analytique simple, la KL voit aussi l’écart entre les moyennes. Le point du notebook est plus précis : le transport optimal mesure cet écart comme un déplacement de masse sur le support. C’est cette lecture géométrique que l’on veut vérifier ici.

In [ ]:
def gaussian_pmf(grid, mu, sigma):
    """Densité gaussienne 1D discrétisée sur une grille, normalisée à somme 1."""
    pdf = np.exp(-0.5 * ((grid - mu) / sigma) ** 2)
    return pdf / pdf.sum()


grid = np.linspace(-6, 12, 200)
sigma = 0.8
shifts = np.array([0.0, 1.0, 2.0, 3.0, 4.0, 5.0])

a_gauss = gaussian_pmf(grid, mu=0.0, sigma=sigma)

# Coût quadratique sur une grille commune : on compare un coût de type W_2^2.
C_1d = (grid[:, None] - grid[None, :]) ** 2

transport_costs = []
for delta in shifts:
    b_gauss = gaussian_pmf(grid, mu=delta, sigma=sigma)
    cost, _ = sinkhorn_transport_cost(
        a_gauss,
        b_gauss,
        C_1d,
        reg=0.05,
    )
    transport_costs.append(cost)

transport_costs = np.array(transport_costs)

assert np.all(np.diff(transport_costs) > 0), "le coût devrait croître avec le décalage"

print("Décalage δ  ->  coût Sinkhorn régularisé   (référence W2² = δ²)")
for delta, cost in zip(shifts, transport_costs):
    print(f"  δ = {delta:.1f}  ->  coût ≈ {cost:6.3f}   (δ² = {delta**2:6.3f})")

bias = transport_costs[0]
debiased_costs = transport_costs - bias

print(f"Biais entropique estimé à δ=0 : {bias:.3f}")
print("Après soustraction de ce biais, la courbe suit presque exactement δ².")

L’offset presque constant vient du biais entropique : même pour $\delta=0$, le plan Sinkhorn régularisé garde un léger flou autour de la diagonale et paie donc un petit coût positif. Comme on translate la même forme sans changer sa largeur, ce coût de flou reste presque constant ; la partie variable du coût suit bien $\delta^2$.

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=shifts,
        y=transport_costs,
        mode="lines+markers",
        name="coût Sinkhorn régularisé",
        line=dict(color="#7c3aed", width=2),
        marker=dict(size=8),
    )
)

fig.add_trace(
    go.Scatter(
        x=shifts,
        y=shifts ** 2,
        mode="lines",
        name="référence continue W₂² = δ²",
        line=dict(color="gray", width=2, dash="dash"),
    )
)

fig.update_layout(
    title="Le coût de transport croît avec la translation",
    xaxis_title="décalage δ entre les deux gaussiennes",
    yaxis_title="coût de transport quadratique",
    template="plotly_white",
    width=700,
    height=450,
)

fig.show()

**Lecture :** la courbe Sinkhorn suit la croissance de la référence $\delta^2$. Elle n’est pas exactement confondue avec elle, pour deux raisons : les gaussiennes sont discrétisées sur une grille finie, et le plan est régularisé par entropie.

Le résultat important est la monotonie. Quand on translate la même masse plus loin, le coût augmente. Le scalaire retourné garde donc la lecture physique du transport : il mesure le travail nécessaire pour déplacer une distribution vers l’autre.

Cette figure ne sert pas à opposer Wasserstein et KL dans tous les cas. Pour deux gaussiennes de même variance, KL est aussi bien définie et sensible au décalage. La différence est dans l’interprétation : KL compare des densités dans un même système de coordonnées, tandis que le transport optimal construit explicitement un déplacement de masse sur le support.

## 7. Vraies données : transporter la masse d’un chiffre MNIST

On passe maintenant à une vraie grille de pixels avec MNIST.

Une image peut être lue comme une distribution de masse : les pixels clairs portent peu de masse, les pixels foncés portent beaucoup de masse. Pour éviter de comparer deux chiffres image par image, on construit d’abord un chiffre moyen par classe. Le "0" moyen donne une masse répartie autour d’un anneau. Le "1" moyen concentre plutôt la masse sur une barre verticale.

La question devient alors très concrète :

$$
\text{où doit partir la masse du 0 moyen pour former le 1 moyen ?}
$$

On utilise MNIST plutôt que `load_digits`, parce que les images $28\times28$ ont une structure visuelle beaucoup plus lisible. Pour garder le calcul et les flèches interprétables, on réduit les images en $14\times14$ par moyenne locale. Ce n’est pas un changement de problème : on transporte toujours une masse de pixels, simplement sur une grille un peu moins fine.

In [ ]:
mnist_train = MNIST(
    root=DATA_DIR,
    train=True,
    download=False,
)

images = mnist_train.data.numpy().astype(np.float64) / 255.0
labels = mnist_train.targets.numpy()

mean_0_full = images[labels == 0].mean(axis=0)
mean_1_full = images[labels == 1].mean(axis=0)


def downsample_2x2(image):
    """Réduit une image 28x28 en 14x14 par moyenne de blocs 2x2."""
    h, w = image.shape
    return image.reshape(h // 2, 2, w // 2, 2).mean(axis=(1, 3))


mean_0 = downsample_2x2(mean_0_full)
mean_1 = downsample_2x2(mean_1_full)

a_mnist = mean_0.ravel()
b_mnist = mean_1.ravel()

a_mnist = a_mnist / a_mnist.sum()
b_mnist = b_mnist / b_mnist.sum()

grid_size = mean_0.shape[0]
rows, cols = np.meshgrid(
    np.linspace(0.0, 1.0, grid_size),
    np.linspace(0.0, 1.0, grid_size),
    indexing="ij",
)
pixel_coords = np.stack([rows.ravel(), cols.ravel()], axis=1)

C_mnist = transport_cost_matrix(pixel_coords, pixel_coords)

cost_mnist, P_mnist = sinkhorn_transport_cost(
    a_mnist,
    b_mnist,
    C_mnist,
    reg=0.01,
    n_iters=800,
)

row_error = np.abs(P_mnist.sum(axis=1) - a_mnist).max()
col_error = np.abs(P_mnist.sum(axis=0) - b_mnist).max()

print(f"Coût Sinkhorn(0 moyen, 1 moyen) : {cost_mnist:.4f}")
print(f"Shape du plan P                 : {P_mnist.shape}")
print(f"max|P.sum(axis=1) - a|          : {row_error:.2e}")
print(f"max|P.sum(axis=0) - b|          : {col_error:.2e}")

In [ ]:
target_center = (P_mnist @ pixel_coords) / P_mnist.sum(axis=1, keepdims=True)
displacement = target_center - pixel_coords

mass_scale = a_mnist / a_mnist.max()
arrow_mask = mass_scale > 0.18

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        'source a : "0" moyen',
        'cible b : "1" moyen',
        "déplacement moyen de la masse",
    ),
    specs=[[{"type": "heatmap"}, {"type": "heatmap"}, {"type": "xy"}]],
    horizontal_spacing=0.08,
)

fig.add_trace(
    go.Heatmap(z=mean_0, colorscale="Gray_r", showscale=False),
    row=1,
    col=1,
)
fig.add_trace(
    go.Heatmap(z=mean_1, colorscale="Gray_r", showscale=False),
    row=1,
    col=2,
)

fig.add_trace(
    go.Heatmap(
        z=mean_0,
        colorscale="Gray_r",
        showscale=False,
        opacity=0.55,
    ),
    row=1,
    col=3,
)

for k, show_arrow in enumerate(arrow_mask):
    if not show_arrow:
        continue

    y0 = pixel_coords[k, 0] * (grid_size - 1)
    x0 = pixel_coords[k, 1] * (grid_size - 1)
    dy = displacement[k, 0] * (grid_size - 1)
    dx = displacement[k, 1] * (grid_size - 1)

    fig.add_annotation(
        x=x0 + dx,
        y=y0 + dy,
        ax=x0,
        ay=y0,
        xref="x3",
        yref="y3",
        axref="x3",
        ayref="y3",
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1.2,
        arrowcolor=f"rgba(220,38,38,{0.25 + 0.65 * mass_scale[k]:.3f})",
    )

for col in [1, 2, 3]:
    fig.update_xaxes(showticklabels=False, row=1, col=col)
    fig.update_yaxes(showticklabels=False, autorange="reversed", scaleanchor=f"x{col}", row=1, col=col)

fig.update_layout(
    width=1050,
    height=420,
    template="plotly_white",
    margin=dict(l=20, r=20, t=70, b=20),
)

fig.show()

**Lecture :** le panneau de droite ne montre pas un appariement pixel à pixel. Chaque flèche part d’un pixel source significatif du "0" moyen et pointe vers le centre de masse des destinations que le plan $P$ lui attribue.

Le mouvement dominant est cohérent avec l’image : une partie de la masse latérale du "0" se déplace vers la zone centrale où le "1" moyen concentre son intensité. Mais le plan reste global. Il ne choisit pas simplement le pixel cible le plus proche pour chaque pixel source. Il doit satisfaire toutes les marginales : chaque pixel source envoie exactement sa masse, et chaque pixel cible reçoit exactement la masse demandée par le "1" moyen.

Le scalaire `cost_mnist` résume le coût total de cette réorganisation. Les flèches montrent la partie interprétable : comment la masse du chiffre source se redistribue pour former la cible.

## 8. Lien génératif : WGAN prend la route duale

Le NB30 a déjà rencontré le mot Wasserstein avec le WGAN-GP. Le lien avec ce notebook est réel, mais il faut le formuler avec précision.

Dans ce notebook, on a pris la route **primale** du transport optimal. On a construit un plan explicite :

$$
P_{ij}
=
\text{masse envoyée de } x_i \text{ vers } y_j,
$$

puis on a calculé un coût :

$$
\langle P,C\rangle.
$$

Cette route est très interprétable. Elle permet de tracer les trajets, de vérifier les marginales, et de voir comment une distribution se réorganise vers une autre. Son défaut est évident : si le support contient $n$ points source et $m$ points cible, le plan $P$ contient $nm$ coefficients. Sur une grille MNIST réduite, c’est encore raisonnable. Sur des images haute résolution, ou sur une distribution continue d’images naturelles, ce n’est plus praticable.

Le WGAN fait autre chose. Il utilise la formulation **duale** de Kantorovich-Rubinstein pour $W_1$ :

$$
W_1(p_{\text{data}},p_g)
=
\sup_{\lVert D\rVert_L\le 1}
\mathbb E_{x\sim p_{\text{data}}}[D(x)]
-
\mathbb E_{x\sim p_g}[D(x)].
$$

Le critic $D$ ne construit jamais de plan de transport. Il apprend plutôt une fonction qui donne un score plus haut aux vrais exemples qu’aux exemples générés, sous une contrainte 1-Lipschitz. Le gradient penalty du WGAN-GP sert à faire respecter approximativement cette contrainte.

Les deux approches parlent donc de transport, mais elles ne résolvent pas le même objet pratique :

| Route | Ce qu’elle calcule | Avantage | Limite |
|---|---|---|---|
| Primale, ce notebook | un plan $P$ et un coût $\langle P,C\rangle$ | interprétable, vérifiable, visuel | coûte $O(nm)$ par itération |
| Duale, WGAN-GP | un estimateur de $W_1$ via un critic | utilisable en haute dimension continue | pas de plan explicite |

Ce notebook utilise souvent un coût quadratique $C_{ij}=\lVert x_i-y_j\rVert^2$ et une régularisation entropique. Il calcule donc un coût de transport Sinkhorn de type $W_2^2$ régularisé. Le WGAN, lui, vise plutôt $W_1$ par le dual. Il ne faut donc pas dire que les deux calculent exactement le même nombre. Il faut dire qu’ils exploitent la même idée géométrique : comparer des distributions par le coût d’un déplacement de masse, au lieu de les comparer seulement case par case.

```mermaid
flowchart TD
    A["Comparer deux distributions"]
    A --> B["Route primale<br/>ce notebook"]
    A --> C["Route duale<br/>WGAN-GP, NB30"]

    B --> B1["Plan explicite P"]
    B1 --> B2["Contraintes<br/>P 1 = a<br/>P^T 1 = b"]
    B2 --> B3["Coût<br/><P,C><br/>souvent régularisé"]

    C --> C1["Critic D 1-Lipschitz"]
    C1 --> C2["Différence de scores<br/>E_data[D] - E_gen[D]"]
    C2 --> C3["Estimateur de W1<br/>sans plan P"]

    classDef default fill:none,stroke:#64748b,stroke-width:1.5px;
    classDef focus fill:none,stroke:#16a34a,stroke-width:2px;
    class B,C,B1,C1 focus;
```

**Lecture :** la route primale construit la carte du transport. La route duale apprend une fonction qui estime le coût sans jamais écrire cette carte. C’est pour cela que Sinkhorn est précieux quand on veut comprendre ou visualiser un transport discret, tandis que WGAN est utile quand le plan explicite serait beaucoup trop grand.

### Quand choisir le transport optimal ?

La formulation primale est un bon choix quand la géométrie du coût est connue,
que les distributions ont une taille modérée et que le plan $P$ doit rester
interprétable. Sinkhorn devient utile quand on accepte un biais entropique en
échange d'un calcul régulier et différentiable.

À éviter quand la matrice de coût $n\times m$ ne tient plus en mémoire ou quand
la distance choisie ne correspond pas au phénomène. Tester alors une
formulation duale comme WGAN, une distance par projections, ou un modèle de
flow si l'on veut apprendre directement une dynamique de transport.


## 9. Limites, choix pratiques et extensions


### Ce qu’on a établi

Le transport optimal ajoute une pièce qui manquait aux comparaisons classiques : la géométrie du support.

Une divergence KL, une cross-entropy ou une distance euclidienne ne sont pas mauvaises. Elles répondent simplement à une autre question. KL demande si deux distributions mettent leur masse sur les mêmes événements. La cross-entropy est parfaite quand on veut placer la probabilité sur la bonne classe. Une distance euclidienne entre deux vecteurs est naturelle quand les coordonnées sont déjà appariées.

Le transport optimal devient utile quand les indices ont une position :

$$
\text{pixels, bins ordonnés, points d'un nuage, prototypes, embeddings}.
$$

Dans ce cas, comparer case par case oublie une information essentielle : une masse déplacée juste à côté n’est pas la même chose qu’une masse déplacée très loin. Le transport optimal mesure ce déplacement.

Dans ce notebook, on a construit :

- une matrice de coût $C$, qui encode la géométrie choisie ;
- un plan de transport $P$, qui dit quelle masse part de chaque source vers chaque cible ;
- les contraintes de marginales $P\mathbf 1=a$ et $P^\top\mathbf 1=b$ ;
- la régularisation entropique, qui rend le problème plus lisse et calculable ;
- Sinkhorn en domaine log, qui garde le calcul stable quand `reg` devient petit ;
- des visualisations du plan sur un nuage synthétique et sur MNIST.

Le point important est que toute la géométrie entre par $C$. Dans nos exemples, le support est simple : ligne 1D, plan 2D, grille de pixels. On a donc utilisé un coût euclidien quadratique. Sur une variété, on pourrait remplacer ce coût par une distance géodésique, une distance de graphe, ou une distance apprise dans un latent. C’est précisément pour cela que ce notebook appartient à la phase géométrie : il ne demande pas seulement combien deux distributions diffèrent, mais combien il faut se déplacer dans l’espace qui les porte.


### Quand utiliser OT

Le transport optimal est un bon candidat quand la comparaison doit respecter une structure spatiale ou géométrique :

| Cas | Pourquoi OT peut aider |
|---|---|
| Histogrammes avec bins ordonnés | déplacer une masse vers un bin voisin doit coûter moins cher que vers un bin lointain |
| Images vues comme distributions de pixels | deux formes légèrement décalées restent proches géométriquement |
| Domain adaptation | comparer deux nuages de représentations source et cible |
| Représentations internes | comparer des distributions d’activations entre classes, couches ou modèles |
| Génératif | comparer $p_{\text{data}}$ et $p_g$ même quand les supports se recouvrent mal |
| JEPA ou modèles prédictifs | utile si l’on compare deux ensembles de latents non parfaitement appariés |

À l’inverse, OT n’est pas le bon réflexe partout. Si chaque prédiction est déjà appariée à une cible précise, une loss L2 ou cosine est souvent plus simple et plus directe. Si l’on compare des classes discrètes sans géométrie naturelle entre labels, la cross-entropy reste le bon outil. Si l’on régularise un posterior de VAE vers un prior gaussien, la KL analytique est plus adaptée.

La règle pratique est donc :

$$
\boxed{
\text{utiliser OT quand la position de la masse fait partie du problème.}
}
$$


### Limites

Le premier coût est informatique. Même avec Sinkhorn, une itération manipule une matrice $P$ ou $C$ de taille $n\times m$. Cela reste très raisonnable pour quelques centaines ou milliers de points, mais pas pour des distributions continues en haute dimension sans approximation.

Le deuxième coût est statistique et numérique. La régularisation entropique rend le calcul plus stable, mais elle ajoute un biais. Même transporter une distribution vers elle-même peut donner un coût légèrement positif, parce que le plan garde un peu de flou autour de la diagonale. Diminuer `reg` réduit ce biais, mais rend les calculs plus difficiles. C’est le compromis central de Sinkhorn.

Une extension importante corrige en partie ce biais : la **Sinkhorn divergence**. Au lieu d’utiliser seulement le coût régularisé entre $a$ et $b$, elle soustrait les coûts à soi-même :

$$
S_\varepsilon(a,b)
=
\mathrm{OT}_\varepsilon(a,b)
-
\frac{1}{2}\mathrm{OT}_\varepsilon(a,a)
-
\frac{1}{2}\mathrm{OT}_\varepsilon(b,b).
$$

Cette correction rend la quantité nulle quand $a=b$, tout en gardant une partie des avantages numériques de Sinkhorn.

D’autres extensions naturelles existent : barycentres de Wasserstein, transport non balancé quand les masses totales diffèrent, coûts géodésiques sur variétés, coûts appris dans un espace latent, ou transport entre distributions de représentations neuronales.


### Pont vers la suite

Le NB40 va comparer des distributions par une autre géométrie : la géométrie de l’information. Au lieu de mesurer un déplacement global de masse, on regardera la métrique de Fisher, locale et différentielle, puis le gradient naturel.

Le contraste est intéressant :

| Notebook | Question géométrique |
|---|---|
| NB39, transport optimal | combien coûte le déplacement global d’une masse ? |
| NB40, information géométrique | quelle est la bonne direction locale dans l’espace des distributions ? |
| NB43, géométrie des représentations | comment comparer et analyser les distributions internes d’un réseau ? |

NB39 donne donc un outil de comparaison globale. NB40 donnera une métrique locale. Les deux racontent la même idée de fond : en machine learning, une distribution n’est pas seulement un vecteur de nombres. Elle vit dans un espace, et cet espace impose une géométrie.


## 10. Version packagée et bibliographie


### Version packagée

Le notebook a reconstruit Sinkhorn pour rendre chaque étape visible. La version à utiliser dans le package est ici :

```python
from ml_from_scratch.geometric.optimal_transport import (
    sinkhorn,
    sinkhorn_plan,
    wasserstein_distance,
    WassersteinDistance,
)
```

Le fichier `src/ml_from_scratch/geometric/optimal_transport/sinkhorn.py` contient la version testée et validée. Elle suit les mêmes conventions que le notebook :

- `a` : masses source ;
- `b` : masses cible ;
- `C` : matrice de coût ;
- `reg` : régularisation entropique ;
- `P` : plan de transport ;
- `f`, `g` : potentiels log-domain.

La fonction `wasserstein_distance` du package renvoie par défaut le coût brut $\langle P,C\rangle$ du plan Sinkhorn. Ce n’est pas automatiquement une distance débiaisée, ni toujours $W_p$ avec racine. Le nom est pratique pour l’API, mais dans une lecture mathématique il faut garder en tête le coût choisi dans $C$ et la présence de la régularisation.


### Articles fondateurs

- **Cuturi, M. (2013)**. *Sinkhorn Distances: Lightspeed Computation of Optimal Transport.* NeurIPS 2013.  
  [https://arxiv.org/abs/1306.0895](https://arxiv.org/abs/1306.0895)  
  Introduit la régularisation entropique du transport optimal et montre pourquoi Sinkhorn rend le calcul beaucoup plus rapide.

- **Peyré, G., & Cuturi, M. (2019)**. *Computational Optimal Transport.* Foundations and Trends in Machine Learning.  
  [https://arxiv.org/abs/1803.00567](https://arxiv.org/abs/1803.00567)  
  Référence complète pour Monge, Kantorovich, Wasserstein, Sinkhorn, les stabilisations numériques et les extensions modernes.

- **Kantorovich, L. V. (1942/2006)**. *On the translocation of masses.* Journal of Mathematical Sciences.  
  [Traduction publique (PDF)](https://www.math.toronto.edu/~mccann/assignments/477/Kantorovich42.pdf) · [DOI](https://www.math.toronto.edu/~mccann/assignments/477/Kantorovich42.pdf)  
  Formulation du transport comme optimisation sur des plans, plutôt que comme fonction déterministe de Monge.

- **Sinkhorn, R., & Knopp, P. (1967)**. *Concerning nonnegative matrices and doubly stochastic matrices.* Pacific Journal of Mathematics.  
  [https://msp.org/pjm/1967/21-2/pjm-v21-n2-p14-p.pdf](https://msp.org/pjm/1967/21-2/pjm-v21-n2-p14-p.pdf)  
  Algorithme de mise à l’échelle alternée des lignes et colonnes, repris ensuite dans le transport optimal entropique.

- **Arjovsky, M., Chintala, S., & Bottou, L. (2017)**. *Wasserstein GAN.* ICML 2017.  
  [https://arxiv.org/abs/1701.07875](https://arxiv.org/abs/1701.07875)  
  Utilise la route duale de Wasserstein pour entraîner des modèles génératifs quand les divergences classiques donnent des gradients pauvres.
